# Week 5: Advanced Ensemble Algorithms
## Notebook 09: Isolation Forest: Path Length Derivation & Anomaly Scoring

### 1. Conceptual Intuition & Theoretical Background
Most anomaly detection algorithms (e.g. One-Class SVM, LOF) construct models of "normal" instances and identify outliers as points outside normal density profiles.

**Isolation Forest (iForest)** turns this paradigm upside down: anomalies are **"few and different"**, making them susceptible to rapid isolation under random partitioning.

The average path length of an unsuccessful search in a Binary Search Tree (BST) represents the equivalent baseline for random tree isolation:
$$c(n) = 2 \left( \ln(n - 1) + 0.5772156649 \right) - \frac{2(n - 1)}{n}$$
Where $0.5772156649$ is Euler's constant $\gamma$.

Anomaly Score for instance $x$ across an ensemble of $T$ isolation trees:
$$s(x, n) = 2^{-\frac{\mathbb{E}(h(x))}{c(n)}}$$
- If $\mathbb{E}(h(x)) \to 0 \implies s \to 1$: Highly anomalous (isolated in very few splits).
- If $\mathbb{E}(h(x)) \to c(n) \implies s \to 0.5$: Normal instance.
- If $\mathbb{E}(h(x)) \to n - 1 \implies s \to 0$: Deeply embedded central cluster.

---

### 2. Scratch Python Implementation: Isolation Tree & Scorer

In [ ]:
import numpy as np

# Euler-Mascheroni Constant
EULER_GAMMA = 0.5772156649

def c_factor(n):
    """Average path length of unsuccessful search in BST."""
    if n <= 1:
        return 1.0
    if n == 2:
        return 1.0
    return 2.0 * (np.log(n - 1.0) + EULER_GAMMA) - (2.0 * (n - 1.0) / n)

class IsolationTreeNode:
    def __init__(self, left=None, right=None, split_feature=None, split_value=None, size=None):
        self.left = left
        self.right = right
        self.split_feature = split_feature
        self.split_value = split_value
        self.size = size

def build_itree(X, current_depth=0, max_depth=10):
    n_samples, n_features = X.shape
    if current_depth >= max_depth or n_samples <= 1:
        return IsolationTreeNode(size=n_samples)
        
    # Select random feature
    feat = np.random.randint(0, n_features)
    f_min, f_max = X[:, feat].min(), X[:, feat].max()
    
    if f_min == f_max:
        return IsolationTreeNode(size=n_samples)
        
    # Select random split point uniformly between min and max
    split_val = np.random.uniform(f_min, f_max)
    
    left_mask = X[:, feat] < split_val
    right_mask = ~left_mask
    
    left_child = build_itree(X[left_mask], current_depth + 1, max_depth)
    right_child = build_itree(X[right_mask], current_depth + 1, max_depth)
    
    return IsolationTreeNode(left=left_child, right=right_child, 
                             split_feature=feat, split_value=split_val, size=n_samples)

def path_length(x, node, current_depth=0):
    """Computes tree depth required to isolate sample x."""
    if node.left is None or node.right is None:
        return current_depth + c_factor(node.size)
        
    if x[node.split_feature] < node.split_value:
        return path_length(x, node.left, current_depth + 1)
    else:
        return path_length(x, node.right, current_depth + 1)

# Generate Normal Gaussian Data + Extreme Outliers
np.random.seed(42)
normal_data = np.random.normal(0, 1, (300, 2))
outliers = np.array([[6.0, 6.0], [-5.5, -4.5], [7.0, -6.0]])

# Build ensemble of 50 Isolation Trees
n_trees = 50
forest = [build_itree(normal_data, max_depth=8) for _ in range(n_trees)]

# Compute average path length and anomaly scores
def compute_anomaly_score(sample, trees, n_pop):
    avg_path = np.mean([path_length(sample, t) for t in trees])
    c_n = c_factor(n_pop)
    score = 2.0 ** (-avg_path / c_n)
    return avg_path, score

avg_p_norm, score_norm = compute_anomaly_score(normal_data[0], forest, len(normal_data))
avg_p_out, score_out = compute_anomaly_score(outliers[0], forest, len(normal_data))

print(f"Normal Point Avg Path Length:  {avg_p_norm:.2f} | Anomaly Score: {score_norm:.4f}")
print(f"Outlier Point Avg Path Length: {avg_p_out:.2f} | Anomaly Score: {score_out:.4f}")
print("Outlier isolated much faster (shorter path), resulting in anomaly score close to 1.0!")

### 3. Step-by-Step Code Walkthrough
1. **`c_factor(n)`**: Calculates theoretical average depth $c(n)$ of a random binary tree for normal data.
2. **Random Cuts**: Each tree picks a random feature and uniform threshold, quickly isolating low-density anomalies.
3. **Score Mapping**: Score $s > 0.6$ indicates anomalies; score $s < 0.5$ indicates standard inliers.

---

### 4. Industry Applications & Trade-offs
- **Credit Card Fraud Detection:** Rapidly detects unauthorized transactions that deviate from normal spending patterns.
- **Trade-off:** Axis-aligned splits can create "ghost anomalies" in empty hypercube corners. Extended Isolation Forest (EIF) resolves this using random hyperplanes.

---

### 5. Top Interview Questions & Answers
**Q1: Why is sub-sampling (e.g. $n = 256$) strongly recommended when building Isolation Forests?**
> **Answer:** Sub-sampling resolves **swamping** (normal instances mistakenly flagged as anomalies) and **masking** (large clusters of anomalies concealing each other), while drastically reducing memory and training time.

**Q2: What is the significance of an anomaly score $s(x, n) = 0.5$?**
> **Answer:** If $s = 0.5$, the average path length $\mathbb{E}(h(x))$ equals $c(n)$ (the average depth of a random binary search tree), indicating the instance exhibits zero anomalous characteristics.